### Notebook to run the simulation of the system end-to-end through terminal inputs

Demo of the full loop. The edge model (V0) predicts the intent of what you type. If the utterance is unknown
(OOD by Energy score) it is escalated to the server, the server uses an LLM to name the new intent and generate
training data, retrains the model into a new version, and the edge loads it and predicts again.

What you need before starting:
- a GPU runtime (the free T4 works)
- an Anthropic API key, the server calls Claude every time a new intent is learned
- nothing else, the code is cloned from GitHub and the V0 model is downloaded from Hugging Face

W&B is not needed, it is disabled below.

Run the cells in order, only the simulation cell asks for input.

#### 1. GPU runtime

Runtime > Change runtime type > T4 GPU, then run:

In [ ]:
import torch

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "not available, change the runtime to GPU before continuing")

#### 2. Anthropic API key

Add it as a Colab secret: key icon on the left sidebar > Add new secret, name `ANTHROPIC_API_KEY`, paste the key
and enable notebook access. If there is no secret the cell asks for the key instead.

In [ ]:
import os
from getpass import getpass

try:
    from google.colab import userdata
    key = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    key = getpass("Anthropic API key: ")

os.environ["ANTHROPIC_API_KEY"] = key.strip()
os.environ["WANDB_MODE"] = "disabled"
os.environ["WANDB_SILENT"] = "true"

#### 3. Clone the repository

In [ ]:
REPO_DIR = "/content/incremental_learning_on_the_edge"
CKPT_DIR = f"{REPO_DIR}/src/outputs/checkpoints/student_ettin_68m_distill_v0"

!git clone -q https://github.com/BrunoSantome/incremental_learning_on_the_edge.git {REPO_DIR} 2>/dev/null || git -C {REPO_DIR} pull -q

#### 4. Install dependencies

The pip warnings about gradio, diffusers or gcsfs can be ignored, those packages are not used here.

In [ ]:
%pip install -q "transformers>=4.48,<5" "huggingface_hub>=0.34,<1" "datasets<3.0.0" anthropic python-dotenv sentencepiece

#### 5. Download the V0 checkpoint

The model is too big for GitHub so it is on Hugging Face:
[Brunaquen/ettin-68m-massive15-v0](https://huggingface.co/Brunaquen/ettin-68m-massive15-v0). It gets downloaded
into `src/outputs/checkpoints/student_ettin_68m_distill_v0/`, which is the folder the code loads V0 from. No HF
account is needed (the `HF_TOKEN` warning can be ignored).

If the download fails, get `config.json`, `model.safetensors` and `history-student1.json` from the Files tab of the
model page and upload them to that folder with the Colab file browser.

In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="Brunaquen/ettin-68m-massive15-v0",
    local_dir=CKPT_DIR,
    allow_patterns=["config.json", "model.safetensors", "history-student1.json"],
)

In [ ]:
# should list config.json, history-student1.json and model.safetensors (~260M)
!ls -lh {CKPT_DIR}

#### 6. Run the simulation

An input box shows up under the cell after the V0 threshold line. Write an utterance and press Enter, write `quit`
to end the run.

Some examples to try:
- `what's the weather like tomorrow` -> known intent, stays on the device
- `open the front window of the car` -> unknown, escalated, the server learns it (V1)
- `roll down the passenger window` -> should now be predicted with the new intent
- `how much battery charge is left` -> unknown again, V2
- `play some jazz` -> known intent, checks that the old intents are not forgotten

Every escalation takes a few minutes on a T4 (LLM generation + retraining). The TensorFlow messages at the start
(`Could not find cuda drivers`, `oneDNN ...`) are not from this project, ignore them.

In [ ]:
%cd {REPO_DIR}/src
!python run_simulation.py

#### 7. Download the results (optional)

Each run saves a `log.json` in `src/outputs/results/e2e_interactive_<date>/` with every utterance, prediction,
energy, escalation, timings and the per-intent F1, plus the generated dataset of each version. Colab deletes
everything when the session ends, so download it if you want to keep it.

In [ ]:
import glob
import shutil
from google.colab import files

last_run = sorted(glob.glob(f"{REPO_DIR}/src/outputs/results/e2e_interactive_*"))[-1]
files.download(shutil.make_archive(last_run, "zip", last_run))

#### If something fails

- error loading `student_ettin_68m_distill_v0`: the checkpoint is not in place, run step 5 again
- `AuthenticationError` or 401 when the server calls the LLM: wrong API key, redo step 2 and run step 6 again
- `credit balance is too low`: the Anthropic account has no credits
- retraining very slow: there is no GPU, see step 1

Running step 6 again always starts from V0, the registry and the mailbox are reset at start-up.

### Pre-executed Run 4

scripted 15-utterance run, the inputs are fixed and reproducible

Shortened. The LLM generation changes between runs, so the intent names and numbers will not be exactly the same.

[1] START-UP: the device computes its own threshold from known-intent data
---

[edge v0] Energy T=2.0 threshold 9.8607 (keeps 95% of known-intent utterances on the device)

[2] DETECT -> LEARN -> REDEPLOY (first acquisition, V0 -> V1)
---

utterance> open the front window of the car

[edge v0] predicted 'weather_query' (confidence 0.74, energy 7.0870 vs threshold 9.8607)

[edge] OOD: escalating

```
Evaluation of the model trained Retention and acquisition

retention (old intents)
                         V0        V1
alarm_set          0.975610  0.962963
audio_volume_down  1.000000  1.000000
audio_volume_mute  0.939394  0.937500
audio_volume_up    0.916667  0.833333
datetime_query     0.960452  0.954023
email_addcontact   0.923077  0.923077
lists_createoradd  0.945946  0.947368
news_query         0.964427  0.960317
play_audiobook     0.805195  0.784810
play_game          0.861538  0.869565
play_music         0.928962  0.940171
play_radio         0.936170  0.931507
transport_query    0.907216  0.929293
transport_taxi     1.000000  1.000000
weather_query      0.980769  0.980645
mean_old           0.936362  0.930305

acquisition (new intents)
                      V0       V1
car_window_control  None  0.77551
mean_new             NaN  0.77551
```

server calibrated v1 threshold: 8.9572

server published v1 (car_window_control)

[edge v1] now predicts 'car_window_control' (confidence 0.99, energy 10.4257 vs threshold 8.9572)

utterance> roll down the passenger window

[edge v1] predicted 'car_window_control' (confidence 1.00, energy 10.8495 vs threshold 8.9572)

[3] ABSORPTION: unknown capability next to a learned one  (F11)
---

utterance> turn on the heating of the driver seat

[edge v1] predicted 'car_window_control' (confidence 0.97, energy 10.4512 vs threshold 8.9572)

[4] COUNTER-EXAMPLE: unknown capability in an empty region IS detected

---

utterance> how much battery charge is left

[edge v1] predicted 'car_window_control' (confidence 0.61, energy 6.7404 vs threshold 8.9572)

[edge] OOD: escalating

v2: production intent 'device_battery_query'

[5] ABSORPTION BY A PRE-EXISTING CLASS + NEAR-OOD PROBE
---

utterance> how many miles can i still drive

[edge v2] predicted 'transport_query' (confidence 1.00, energy 9.7917 vs threshold 8.5207)

utterance> remove milk from my shopping list

[edge v2] predicted 'lists_createoradd' (confidence 1.00, energy 10.7171 vs threshold 8.5207)

[6] FALSE ESCALATION -> DUPLICATE CLASS  (F12)
---

utterance> say something funny

[edge v3] predicted 'humor_tell_joke' (confidence 0.98, energy 7.2861 vs threshold 7.7753)

[edge] OOD: escalating

```
Evaluation of the model trained Retention and acquisition

retention (old intents)
                         V0        V1        V2        V3        V4
alarm_set          0.975610  0.962963  0.975610  0.987952  0.975610
audio_volume_down  1.000000  1.000000  1.000000  1.000000  1.000000
audio_volume_mute  0.939394  0.937500  0.968750  0.968750  0.968750
audio_volume_up    0.916667  0.833333  0.869565  0.869565  0.880000
datetime_query     0.960452  0.954023  0.960000  0.965517  0.965517
email_addcontact   0.923077  0.923077  0.923077  0.923077  0.923077
lists_createoradd  0.945946  0.947368  0.947368  0.933333  0.933333
news_query         0.964427  0.960317  0.971660  0.967742  0.962963
play_audiobook     0.805195  0.784810  0.790123  0.795181  0.767442
play_game          0.861538  0.869565  0.852941  0.845070  0.805556
play_music         0.928962  0.940171  0.934473  0.920821  0.913947
play_radio         0.936170  0.931507  0.937931  0.944444  0.944444
transport_query    0.907216  0.929293  0.929293  0.929293  0.918367
transport_taxi     1.000000  1.000000  1.000000  1.000000  1.000000
weather_query      0.980769  0.980645  0.980519  0.980519  0.963696
mean_old           0.936362  0.930305  0.936087  0.935418  0.928180

acquisition (new intents)
                        V0       V1        V2        V3        V4
car_window_control    None  0.77551  0.851064  0.888889  0.888889
device_battery_query  None      NaN  0.833333  0.869565  0.784314
humor_tell_joke       None      NaN       NaN  0.800000  0.705882
humor_be_funny        None      NaN       NaN       NaN  0.727273
mean_new               NaN  0.77551  0.842199  0.852818  0.776589
server calibrated v4 threshold: 7.5056
```

[7] ABSORPTION BY A DIFFERENT NEIGHBOURHOOD
---

utterance> check the tyre pressure

[edge v4] predicted 'device_battery_query' (confidence 0.96, energy 9.7910 vs threshold 7.5056)